### OSSERVAZIONE: Scelta del max_length

Prendiamo il 99° percentile, per tagliare una piccolissima parte delle frasi lunghe

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Assumiamo che 'df' sia il tuo DataFrame con la colonna 'text_cleaned'
# df = pd.read_csv(...)
df = preprocess_dataset('goemotions_1.csv')

# Carica lo stesso tokenizer che usi per il modello
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Calcola il numero di token per ogni frase
token_lengths = [len(tokenizer.encode(text)) for text in df['text']]

# Crea una Serie pandas per analizzare le lunghezze
lengths_series = pd.Series(token_lengths)

# Stampa le statistiche chiave, inclusi i percentili
print("Statistiche sulla lunghezza in TOKEN:")
print(lengths_series.describe(percentiles=[.90, .95, .99]))

NameError: name 'preprocess_dataset' is not defined

## Osservazione: perchè usiamo Weighted Binary Cross-Entropy

In [4]:
emotion_columns = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

# Select the emotion columns
emotion_df = goemotions_df[emotion_columns]

# Calculate the sum of each emotion column (counts of each emotion)
emotion_counts = emotion_df.sum()

# Display the counts
print("Counts of each emotion:")
display(emotion_counts)

NameError: name 'goemotions_df' is not defined

# Preprocessing

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
run_num = 1
model_name = "Bert" # o "bert-base-uncased", etc.
dataset_name = "emobank"
base_dir = "/content/drive/MyDrive/tesi/dimensional_model"
# 2. Costruisci il path dinamicamente
dinamic_path = f"{base_dir}/models/best_model_run{run_num}_{model_name}_{dataset_name}"


In [ ]:
import pandas as pd
import re

def preprocess_dataset(path):
    """ Remove unnecessary characters and encode the sentiment labels.

    The type of preprocessing required changes based on the dataset. For the
    IMDb dataset, the review texts contains HTML break tags (<br/>) leftover
    from the scraping process, and some unnecessary whitespace, which are
    removed. Finally, encode the sentiment labels as 0 for "negative" and 1 for
    "positive". This method assumes the dataset file contains the headers
    "review" and "sentiment".

    Parameters:
        path (str): A path to a dataset file containing the sentiment analysis
            dataset. The structure of the file should be as follows: one column
            called "review" containing the review text, and one column called
            "sentiment" containing the ground truth label. The label options
            should be "negative" and "positive".

    Returns:
        df_dataset (pd.DataFrame): A DataFrame containing the raw data
            loaded from the self.dataset path. In addition to the expected
            "review" and "sentiment" columns, are:

            > review_cleaned - a copy of the "review" column with the HTML
                break tags and unnecessary whitespace removed

            > sentiment_encoded - a copy of the "sentiment" column with the
                "negative" values mapped to 0 and "positive" values mapped
                to 1
    """
    df_dataset = pd.read_csv(path)

    # 0. Gestisci i valori mancanti e crea una nuova colonna pulita
    #    Uso .lower() per rendere il testo in minuscolo, dal momento che utilizziamo un bert-base-uncased. Quest'ultimo è stato creato usando solo ed esclusivamente testo già convertito in minuscolo. Se gli dai in input una parola con una lettera maiuscola, per lui è come una parola sconosciuta o straniera. Non la trova nel suo dizionario e quindi non la capisce.
    df_dataset['text'] = df_dataset['text'].fillna('').str.lower()

    # 1. Rimuovi TUTTI i tag HTML (<>) in un colpo solo
    df_dataset['text'] = df_dataset['text'].str.replace(r'<.*?>', ' ', regex=True)

    # 2. Rimuovi TUTTI gli URL in un colpo solo
    df_dataset['text'] = df_dataset['text'].str.replace(r'https?://\S+|www\.\S+', ' ', regex=True)

    # 3. Rimuovi caratteri non essenziali (emoji, simboli strani, ecc.)
    df_dataset['text'] = df_dataset['text'].str.replace(r'[^a-z0-9\s.,!?\'"]', '', regex=True)

    # 4. Normalizza tutti gli spazi, a capo e tab in un singolo spazio
    df_dataset['text'] = df_dataset['text'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # 5. Rimuovi le virgolette
    df_dataset['text'] = df_dataset['text'].str.replace('"', ' ', regex=False) # Qui regex non serve

    # 6. Creo un'unica colonna di labels per Valence, Arousal, Dominance
    df_dataset['labels'] = df_dataset[['V', 'A', 'D']].values.tolist()

    return df_dataset

# Fine-Tuning Pipeline Class: BERT and emobank

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, random_split
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    get_linear_schedule_with_warmup)

class FineTuningPipeline:

    def __init__(
            self,
            df_train,
            df_val,
            tokenizer,
            model,
            optimizer,
            save_path, # <--- NUOVO PARAMETRO
            loss_function = nn.MSELoss(),
            epochs = 4,
            seed = 42):

        self.df_train = df_train
        self.df_val = df_val
        self.tokenizer = tokenizer
        self.model = model
        self.optimizer = optimizer
        self.save_path = save_path # <--- SALVATO IN SELF
        self.loss_function = loss_function
        self.epochs = epochs
        self.seed = seed

        if torch.cuda.is_available():
            self.device = torch.device('cuda:0')
        else:
            self.device = torch.device('cpu')

        self.model.to(self.device)
        self.set_seeds()

        self.train_token_ids, self.train_attention_masks, self.train_labels = self.tokenize_dataset(self.df_train)
        self.val_token_ids, self.val_attention_masks, self.val_labels = self.tokenize_dataset(self.df_val)

        self.train_dataloader, self.val_dataloader = self.create_dataloaders()
        self.scheduler = self.create_scheduler()
        self.fine_tune()

    def tokenize(self, text):
        batch_encoder = self.tokenizer.encode_plus(
            text,
            max_length = 38,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt')

        token_ids = batch_encoder['input_ids']
        attention_mask = batch_encoder['attention_mask']

        return token_ids, attention_mask

    def tokenize_dataset(self, df):
        token_ids = []
        attention_masks = []

        for review in df['text']:
            tokens, masks = self.tokenize(review)
            token_ids.append(tokens)
            attention_masks.append(masks)

        token_ids = torch.cat(token_ids, dim=0)
        attention_masks = torch.cat(attention_masks, dim=0)
        labels = torch.tensor(df[['V', 'A', 'D']].values, dtype=torch.float32)

        return token_ids, attention_masks, labels

    def create_dataloaders(self):
        train_dataset = TensorDataset(self.train_token_ids, self.train_attention_masks, self.train_labels)
        val_dataset = TensorDataset(self.val_token_ids, self.val_attention_masks, self.val_labels)

        train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16)
        val_dataloader = DataLoader(val_dataset, batch_size=16)

        return train_dataloader, val_dataloader

    def create_scheduler(self):
        num_training_steps = self.epochs * len(self.train_dataloader)
        num_warmup_steps = int(0.1 * num_training_steps)
        scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps)

        return scheduler

    def set_seeds(self):
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        torch.cuda.manual_seed_all(self.seed)

    def fine_tune(self):
        loss_dict = {'epoch': [], 'train_loss': [], 'val_loss': []}

        # MODIFICA 1: Inizializzo best_val_ccc a -infinito perché vogliamo massimizzarlo
        best_val_ccc = -float('inf')
        t0_train = datetime.now()

        for epoch in range(self.epochs):
            # --- Train ---
            self.model.train()
            training_loss = 0

            print(f'{"-"*20} Epoch {epoch+1} {"-"*20}')

            for batch in self.train_dataloader:
                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                self.model.zero_grad()
                outputs = self.model(batch_token_ids, attention_mask=batch_attention_mask, return_dict=True)

                loss = self.loss_function(outputs.logits, batch_labels) # Qui userà CCCLoss se gliela passi
                training_loss += loss.item()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()

            avg_train_loss = training_loss / len(self.train_dataloader)

            # --- Validation ---
            self.model.eval()
            val_loss = 0
            keys = ['mae', 'mae_V', 'mae_A', 'mae_D', 'pearson', 'ccc', 'ccc_V', 'ccc_A', 'ccc_D']
            total_metrics = {k: 0.0 for k in keys}

            for batch in self.val_dataloader:
                batch_token_ids = batch[0].to(self.device)
                batch_attention_mask = batch[1].to(self.device)
                batch_labels = batch[2].to(self.device)

                with torch.no_grad():
                    outputs = self.model(batch_token_ids, attention_mask=batch_attention_mask)
                    loss = self.loss_function(outputs.logits, batch_labels)

                val_loss += loss.item()

                # Metriche
                logits = outputs.logits.detach().cpu().numpy()
                labels = batch_labels.cpu().numpy()
                batch_metrics = self.calculate_regression_metrics(logits, labels)
                for k in total_metrics: total_metrics[k] += batch_metrics[k]

            avg_val_loss = val_loss / len(self.val_dataloader)
            avg_metrics = {k: v / len(self.val_dataloader) for k, v in total_metrics.items()}

            print(f"Loss Train: {avg_train_loss:.4f} | Loss Val: {avg_val_loss:.4f}")
            print(f"CCC Tot: {avg_metrics['ccc']:.4f} (V: {avg_metrics['ccc_V']:.2f}, A: {avg_metrics['ccc_A']:.2f}, D: {avg_metrics['ccc_D']:.2f})")
            print(f"MAE Tot: {avg_metrics['mae']:.4f}")

            # MODIFICA 2: Logica di salvataggio basata su CCC
            if avg_metrics['ccc'] > best_val_ccc:
                print(f"--> Nuovo record CCC! {best_val_ccc:.4f} -> {avg_metrics['ccc']:.4f}. Salvo.")
                best_val_ccc = avg_metrics['ccc']
                self.model.save_pretrained(self.save_path)
                self.tokenizer.save_pretrained(self.save_path)

            print()

        print(f'Total time: {datetime.now()-t0_train}')

    def calculate_regression_metrics(self, preds, labels):
        metrics = {}
        dims = ['V', 'A', 'D']

        # --- 1. MAE (Mean Absolute Error) - Lower is better ---
        metrics['mae'] = mean_absolute_error(labels, preds)
        metrics['mae_V'] = mean_absolute_error(labels[:, 0], preds[:, 0])
        metrics['mae_A'] = mean_absolute_error(labels[:, 1], preds[:, 1])
        metrics['mae_D'] = mean_absolute_error(labels[:, 2], preds[:, 2])

        # --- 2. Pearson & CCC - Higher is better ---
        total_pearson = 0
        total_ccc = 0

        for i, dim in enumerate(dims):
            y_true = labels[:, i]
            y_pred = preds[:, i]

            # Pearson Correlation
            # np.corrcoef restituisce la matrice di covarianza normalizzata. [0,1] è la correlazione tra x e y
            pearson = np.corrcoef(y_true, y_pred)[0, 1]
            if np.isnan(pearson): pearson = 0.0 # Gestione casi limite (es. varianza 0)

            metrics[f'pearson_{dim}'] = pearson
            total_pearson += pearson

            # Concordance Correlation Coefficient (CCC)
            mean_true = np.mean(y_true)
            mean_pred = np.mean(y_pred)
            var_true = np.var(y_true)
            var_pred = np.var(y_pred)
            std_true = np.std(y_true)
            std_pred = np.std(y_pred)

            numerator = 2 * pearson * std_true * std_pred
            denominator = var_true + var_pred + (mean_true - mean_pred)**2

            if denominator == 0:
                ccc = 0.0
            else:
                ccc = numerator / denominator

            metrics[f'ccc_{dim}'] = ccc
            total_ccc += ccc

        # Medie generali
        metrics['pearson'] = total_pearson / 3
        metrics['ccc'] = total_ccc / 3

        return metrics

In [ ]:
class CCCLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super(CCCLoss, self).__init__()
        self.eps = eps

    def forward(self, x, y):
        # x: predizioni (batch_size, 3)
        # y: labels (batch_size, 3)

        # Calcoliamo le statistiche per ogni dimensione (V, A, D)
        vx = x - torch.mean(x, dim=0, keepdim=True)
        vy = y - torch.mean(y, dim=0, keepdim=True)

        rho = torch.sum(vx * vy, dim=0) / (torch.sqrt(torch.sum(vx ** 2, dim=0)) * torch.sqrt(torch.sum(vy ** 2, dim=0)) + self.eps)

        x_std = torch.std(x, dim=0)
        y_std = torch.std(y, dim=0)

        x_var = torch.var(x, dim=0)
        y_var = torch.var(y, dim=0)

        x_mean = torch.mean(x, dim=0)
        y_mean = torch.mean(y, dim=0)

        numerator = 2 * rho * x_std * y_std
        denominator = x_var + y_var + (x_mean - y_mean) ** 2

        ccc = numerator / (denominator + self.eps)

        # La loss è 1 - CCC (perché vogliamo massimizzare CCC, quindi minimizzare 1-CCC)
        # Facciamo la media su V, A, D
        loss = 1 - torch.mean(ccc)

        return loss

# Fine tuning EMOBANK Dataset:

num labels = 3 (V, A e D)

In [ ]:
run_num = 1
model_name = "bert" # o "bert-base-uncased", etc.
dataset_name = "emobank"
base_dir = "/content/drive/MyDrive/tesi/dimensional_model"

# 2. Costruisci il path dinamicamente
dinamic_path = f"{base_dir}/models/best_model_run{run_num}_{model_name}_{dataset_name}"

# Initialise parameters
path = "/content/drive/MyDrive/tesi/dimensional_model/data/emobank.csv"
df_dataset = preprocess_dataset(path)

# --- NUOVA DIVISIONE BASATA SUGLI SPLIT PREDEFINITI ---
# Dividiamo il DataFrame in base alla colonna 'split'
df_train = df_dataset[df_dataset['split'] == 'train'].copy()
df_val = df_dataset[df_dataset['split'] == 'dev'].copy()
df_test = df_dataset[df_dataset['split'] == 'test'].copy() # Questo lo useremo dopo per il test finale

print(f"Dati caricati: {len(df_dataset)} righe")
print(f"Training set:   {len(df_train)} righe (dallo split 'train')")
print(f"Validation set: {len(df_val)} righe (dallo split 'dev')")
print(f"Test set:       {len(df_test)} righe (dallo split 'test')")
print("-" * 30)
# --- FINE NUOVA DIVISIONE ---


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3)
optimizer = AdamW(model.parameters(), lr=2e-5)

criterion = CCCLoss() # Istanzia la nuova loss

# --- MODIFICATO: Passiamo i DataFrame separati ---
fine_tuned_model = FineTuningPipeline(
    df_train = df_train, # Passiamo il training set
    df_val = df_val,     # Passiamo il validation set
    tokenizer = tokenizer,
    model = model,
    optimizer = optimizer,
    # val_size = 0.1, # Non più necessario
    loss_function = criterion, # Passa CCCLoss qui invece di MSELoss
    epochs = 8,
    seed = 42,
    save_path=dinamic_path
    )

# Salva df_test in una variabile globale o in un file per accedervi nella cella di test
# La memorizzazione in una variabile è più semplice se il kernel non viene riavviato.
%store df_test
print("Test set (df_test) salvato in memoria per la prossima cella.")

Dati caricati: 10062 righe
Training set:   8062 righe (dallo split 'train')
Validation set: 1000 righe (dallo split 'dev')
Test set:       1000 righe (dallo split 'test')
------------------------------


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


-------------------- Epoch 1 --------------------
Loss Train: 0.8528 | Loss Val: 0.6818
CCC Tot: 0.3160 (V: 0.61, A: 0.07, D: 0.27)
MAE Tot: 0.3717
--> Nuovo record CCC! -inf -> 0.3160. Salvo.

-------------------- Epoch 2 --------------------
Loss Train: 0.4893 | Loss Val: 0.5186
CCC Tot: 0.4801 (V: 0.64, A: 0.46, D: 0.34)
MAE Tot: 0.2080
--> Nuovo record CCC! 0.3160 -> 0.4801. Salvo.

-------------------- Epoch 3 --------------------
Loss Train: 0.3779 | Loss Val: 0.5061
CCC Tot: 0.4926 (V: 0.67, A: 0.48, D: 0.33)
MAE Tot: 0.1828
--> Nuovo record CCC! 0.4801 -> 0.4926. Salvo.

-------------------- Epoch 4 --------------------
Loss Train: 0.3025 | Loss Val: 0.5056
CCC Tot: 0.4932 (V: 0.66, A: 0.47, D: 0.34)
MAE Tot: 0.1844
--> Nuovo record CCC! 0.4926 -> 0.4932. Salvo.

-------------------- Epoch 5 --------------------
Loss Train: 0.2501 | Loss Val: 0.5141
CCC Tot: 0.4846 (V: 0.66, A: 0.46, D: 0.33)
MAE Tot: 0.1774

-------------------- Epoch 6 --------------------
Loss Train: 0.2150 

## Test set on the model

In [ ]:
dinamic_path = "/content/drive/MyDrive/tesi/dimensional_model/models/best_model_run1_bert_emobank"

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# Funzione helper per il CCC (usando NumPy per velocità su tutto il test set)
def calculate_ccc_numpy(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    std_true = np.std(y_true)
    std_pred = np.std(y_pred)

    # Pearson via numpy
    pearson = np.corrcoef(y_true, y_pred)[0, 1]

    numerator = 2 * pearson * std_true * std_pred
    denominator = var_true + var_pred + (mean_true - mean_pred)**2

    if denominator == 0:
        return 0.0
    return numerator / denominator

# --- PASSAGGIO 1: CARICA IL MODELLO MIGLIORE E IL TOKENIZER ---
print("Caricamento del modello e del tokenizer salvati...")
# Assumo che 'dinamic_path' sia definito nella cella precedente
tokenizer = AutoTokenizer.from_pretrained(dinamic_path)
model = BertForSequenceClassification.from_pretrained(dinamic_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f"Modello caricato su {device}.")

# --- PASSAGGIO 2: CARICA E PREPARA IL TEST SET ---
try:
    %store -r df_test
    print(f"Test set (df_test) caricato dalla memoria: {len(df_test)} righe.")
except KeyError:
    print("Errore: df_test non trovato. Assicurati di aver eseguito il notebook precedente.")

# --- PASSAGGIO 3: TOKENIZZA E CREA IL DATALOADER DI TEST ---
print("Tokenizzazione del test set...")
test_token_ids = []
test_attention_masks = []

# Nota: per dataset molto grandi questo ciclo è lento, ma per test set normali va bene
for text in df_test['text']:
    batch_encoder = tokenizer.encode_plus(
        text,
        max_length = 38,
        padding = 'max_length',
        truncation = True,
        return_tensors = 'pt'
    )
    test_token_ids.append(batch_encoder['input_ids'])
    test_attention_masks.append(batch_encoder['attention_mask'])

test_token_ids = torch.cat(test_token_ids, dim=0)
test_attention_masks = torch.cat(test_attention_masks, dim=0)
test_labels = torch.tensor(df_test[['V', 'A', 'D']].values, dtype=torch.float32)

test_dataset = TensorDataset(test_token_ids, test_attention_masks, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=32)
print("Dataloader di test creato.")

# --- PASSAGGIO 4: ESEGUI LA PREDIZIONE SUL TEST SET ---
print("Inizio valutazione sul Test Set...")
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch_token_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device) # Importante per coerenza, anche se non usiamo la loss qui

        outputs = model(
            batch_token_ids,
            token_type_ids=None,
            attention_mask=batch_attention_mask
        )

        logits = outputs.logits
        all_preds.append(logits.cpu().numpy())
        all_labels.append(batch_labels.cpu().numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# --- PASSAGGIO 5: CALCOLA E STAMPA LE METRICHE FINALI ---

# 1. MAE
mae_v = mean_absolute_error(all_labels[:, 0], all_preds[:, 0])
mae_a = mean_absolute_error(all_labels[:, 1], all_preds[:, 1])
mae_d = mean_absolute_error(all_labels[:, 2], all_preds[:, 2])
mae_total = mean_absolute_error(all_labels, all_preds)

# 2. Pearson
corr_v, _ = pearsonr(all_labels[:, 0], all_preds[:, 0])
corr_a, _ = pearsonr(all_labels[:, 1], all_preds[:, 1])
corr_d, _ = pearsonr(all_labels[:, 2], all_preds[:, 2])
corr_mean = np.mean([corr_v, corr_a, corr_d])

# 3. CCC (Nuovo)
ccc_v = calculate_ccc_numpy(all_labels[:, 0], all_preds[:, 0])
ccc_a = calculate_ccc_numpy(all_labels[:, 1], all_preds[:, 1])
ccc_d = calculate_ccc_numpy(all_labels[:, 2], all_preds[:, 2])
ccc_mean = np.mean([ccc_v, ccc_a, ccc_d])

print("\n" + "="*40)
print("     RISULTATI FINALI SUL TEST SET     ")
print("="*40)
print(f"{'Metrica':<10} | {'Media':<8} | {'Valence':<8} | {'Arousal':<8} | {'Dominance':<8}")
print("-" * 56)
print(f"{'MAE':<10} | {mae_total:.4f}   | {mae_v:.4f}   | {mae_a:.4f}   | {mae_d:.4f}")
print(f"{'Pearson':<10} | {corr_mean:.4f}   | {corr_v:.4f}   | {corr_a:.4f}   | {corr_d:.4f}")
print(f"{'CCC':<10} | {ccc_mean:.4f}   | {ccc_v:.4f}   | {ccc_a:.4f}   | {ccc_d:.4f}")
print("-" * 56)

Caricamento del modello e del tokenizer salvati...
Modello caricato su cuda.
Test set (df_test) caricato dalla memoria: 1000 righe.
Tokenizzazione del test set...
Dataloader di test creato.
Inizio valutazione sul Test Set...

     RISULTATI FINALI SUL TEST SET     
Metrica    | Media    | Valence  | Arousal  | Dominance
--------------------------------------------------------
MAE        | 0.1816   | 0.1797   | 0.1966   | 0.1686
Pearson    | 0.6091   | 0.7872   | 0.5429   | 0.4973
CCC        | 0.6042   | 0.7779   | 0.5394   | 0.4953
--------------------------------------------------------


# Creiamo english_document_bert_vad.xlxl

In [2]:
dinamic_path = '/content/drive/MyDrive/tesi/dimensional_model/models/best_model_run1_bert_emobank'

In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import re

# --- PASSAGGIO 1: CARICA IL MODELLO MIGLIORE E IL TOKENIZER ---
print("Caricamento del modello VAD (EmoBank) e del tokenizer...")

# --- CORREZIONE CRITICA ---
# Hai bisogno del modello VAD (regressione) che hai appena addestrato,
# non del vecchio modello di classificazione GoEmotions.

tokenizer = AutoTokenizer.from_pretrained(dinamic_path)
model = BertForSequenceClassification.from_pretrained(dinamic_path)

# Imposta il modello in modalità valutazione e spostalo sulla GPU se disponibile
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print("Modello caricato e pronto.")


new_df = pd.read_excel('/content/drive/MyDrive/tesi/analysis/data/english_documents_cleaned.xlsx')


# --- PASSAGGIO 3: TOKENIZZA I NUOVI DATI IN MODO EFFICIENTE ---

def tokenize_for_inference(texts, tokenizer, max_len=64):
    """
    Tokenizza una lista di testi in modo efficiente per l'inferenza.
    """
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=258,
        return_tensors='pt'
    )

print("Tokenizzazione in corso...")
# Usiamo la max_length 38 che era ottimale per EmoBank
encoded_texts = tokenize_for_inference(
    new_df['text'].tolist(),
    tokenizer,
    max_len=256
)

# --- PASSAGGIO 4: CREA UN DATALOADER PER LA PREDIZIONE ---
prediction_dataset = TensorDataset(encoded_texts['input_ids'], encoded_texts['attention_mask'])
prediction_dataloader = DataLoader(prediction_dataset, batch_size=32)

# --- PASSAGGIO 5: ESEGUI LA PREDIZIONE ---
print("Esecuzione dell'inferenza (VAD)...")
all_predictions = []

with torch.no_grad():
    for batch in prediction_dataloader:
        batch_input_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)

        outputs = model(batch_input_ids, attention_mask=batch_attention_mask)
        logits = outputs.logits  # Questo ora contiene i valori V, A, D
        all_predictions.append(logits.cpu().numpy())

predictions_array = np.vstack(all_predictions)
print("Inferenza completata.")

# --- PASSAGGIO 6: AGGIUNGI I RISULTATI AL DATAFRAME ---
new_df['V_predicted'] = predictions_array[:, 0]
new_df['A_predicted'] = predictions_array[:, 1]
new_df['D_predicted'] = predictions_array[:, 2]

print("\n--- RISULTATO FINALE (ANTEPRIMA) ---")
print(new_df[['text', 'V_predicted', 'A_predicted', 'D_predicted']].head())

# --- PASSAGGIO 7: SALVA IL FILE FILTRATO ---

# Definisci la lista esatta delle colonne che vuoi conservare
# Assumo che 'y' sia una colonna ID nel tuo file originale
colonne_da_salvare = [
    'text',
    'V_predicted',
    'A_predicted',
    'D_predicted'
]

# Filtra per le colonne che esistono prima di salvarle
colonne_finali = [col for col in colonne_da_salvare if col in new_df.columns]
df_filtrato = new_df[colonne_finali]

# Definisci il percorso di output
path_output_filtrato = f"/content/drive/MyDrive/tesi/dimensional_model/output/english_documents_5_bert_emobank.xlsx"

# Salva il DataFrame filtrato
df_filtrato.to_excel(path_output_filtrato, index=False)

print(f"\nDataFrame filtrato salvato con successo in: {path_output_filtrato}")
print("\nAnteprima del file salvato:")
print(df_filtrato.head())

Caricamento del modello VAD (EmoBank) e del tokenizer...
Modello caricato e pronto.
Tokenizzazione in corso...
Esecuzione dell'inferenza (VAD)...
Inferenza completata.

--- RISULTATO FINALE (ANTEPRIMA) ---
                                                text  V_predicted  \
0      not having a steady job and being paid little     2.530688   
1  i agree with this definition: we speak of job ...     2.322246   
2  precariousness is the lack of resources in thi...     2.495648   
3  for me precariousness is the fact of not being...     2.543804   
4                                     being helpless     2.117811   

   A_predicted  D_predicted  
0     2.726626     2.546497  
1     3.091984     2.514226  
2     2.985747     2.642273  
3     2.978941     2.551643  
4     3.261431     2.617554  

DataFrame filtrato salvato con successo in: /content/drive/MyDrive/tesi/dimensional_model/output/english_documents_5_bert_emobank.xlsx

Anteprima del file salvato:
                                  